In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import shutil

output_path = "/content/drive/MyDrive/Experiment_3"

if os.path.exists(output_path):
    for item in os.listdir(output_path):
        item_path = os.path.join(output_path, item)

        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
        else:
            os.remove(item_path)

print("Old stored results cleared.")

In [ ]:
import os
import glob
import numpy as np
import cv2
def padding(image,filter):
  f = filter.shape[0]
  paddedimage = np.pad(image,f//2,mode='constant')
  return paddedimage
def extractfilter(image,filtername):
 if filtername=="mean":
    meanfilter = 1/9*np.array([[1,1,1],[1,1,1],[1,1,1]],dtype=float)
    return meanfilter


 elif filtername=="prewhorizontal":
  horfil = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
])
  return horfil
 elif filtername=="prewvertical":
  verfil = np.array([
    [-1,  0,  1],
    [-1,  0,  1],
    [-1,  0,  1]
])
  return verfil
 elif filtername=="sobelhorizontal":
  horfil = np.array([[-1,0,1],[-2,0,2],[-1,0,1]],dtype=float)
  return horfil
 elif filtername=="sobelvertical":
  verfil = np.array([[-1,-2,-1],[0,0,0],[1,2,1]],dtype=float)
  return verfil
 elif filtername=="sobeldiagonal":
  horfil = np.array([[-2,-1,0],[-1,0,1],[0,1,2]],dtype=float)
  return horfil
 elif filtername=="laplacian":
  verfil = np.array([[0,-1,0],[-1,4,-1],[0,-1,0]],dtype=float)
  return verfil
 elif filtername=="composite":
  horfil = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]],dtype=float)
  return horfil
 elif filtername=="laplacianofgaussian":
  verfil = np.array([
    [ 0,  0, -1,  0,  0],
    [ 0, -1, -2, -1,  0],
    [-1, -2, 16, -2, -1],
    [ 0, -1, -2, -1,  0],
    [ 0,  0, -1,  0,  0]
],dtype=float)
  return verfil
 elif filtername=="gaussianblur":
  horfil = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=float) / 16
  return horfil
def medianconvo(image,filter):
  f = filter.shape[0]
  newimage = np.zeros((image.shape[0]-f+1,image.shape[1]-f+1))
  for i in range(image.shape[0]-f+1):
    for j in range(image.shape[1]-f+1):
      window = image[i:i+f,j:j+f]
      median = np.median(window)
      newimage[i,j] = median
  return newimage
def convolution(image,filter):
  l = image.shape[0]
  w = image.shape[1]

  f = filter.shape[0]
  if l<filter.shape[0] or w<filter.shape[1]:
    return None
  else:
    convo = []
    convolvedimage = np.zeros((l-f+1,w-f+1))
    for i in range(l- f + 1):
      for j in range(w-f+1):
        sum = 0
        for m in range(f):
          for n in range(f):
            sum += image[i+m][j+n]*filter[m][n]
        convo.append(sum)
    k=0
    for i in range(l-f+1):
      for j in range(w-f+1):
         convolvedimage[i][j] = convo[k]
         k=k+1
    return convolvedimage

In [ ]:

output_path = "/content/drive/MyDrive/Experiment_3"
os.makedirs(output_path, exist_ok=True)

In [ ]:
def to_displayable(arr):
    # Convert to float for safe calculations
    arr = arr.astype(np.float64)

    # Shift minimum to 0
    arr = arr - arr.min()

    # Scale to [0, 255]
    max_val = arr.max()

    if max_val > 0:
        arr = (arr / max_val) * 255

    # Convert to uint8
    return arr.astype(np.uint8)

In [ ]:
import os
import glob
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

# Path where your images are stored
input_path = "/content"

# Extract all JPG images from /content
image_paths = glob.glob(os.path.join(input_path, "*.jpg"))
print("Images found:", len(image_paths))
k=0
# Read and extract each image
for path in image_paths:
    k = k+1
    print(k)
    name = os.path.splitext(os.path.basename(path))[0]
    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    filter_names = [
    "mean",
    "median",
    "prewhorizontal",
    "prewvertical",
    "sobelhorizontal",
    "sobelvertical",
    "sobeldiagonal",
    "laplacian",
    "composite",
    "laplacianofgaussian",
    "gaussianblur"
]
    if image is None:
        print("Could not read:", path)
        continue
    for filt in filter_names:
      if filt == "median":
        filter_folder = os.path.join(output_path, filt)
        os.makedirs(filter_folder, exist_ok=True)
        nullfilter = np.zeros((3,3))
        padd = padding(image,nullfilter)
        convolvedimage = medianconvo(padd,nullfilter)
        save_path = os.path.join(
    filter_folder,
    name + "_" + filt + ".jpg"
)
        if filt in ["mean", "median", "gaussianblur"]:
        # These are intensity images → clip to [0, 255]
           display_image = np.clip(convolvedimage, 0, 255).astype(np.uint8)
        else:
        # Edge filters → preserve negative and positive responses
        # by normalizing for display
           display_image = to_displayable(convolvedimage)

        cv2.imwrite(save_path, display_image)
        cv2_imshow(display_image)
      else:
       filter_folder = os.path.join(output_path, filt)
       os.makedirs(filter_folder, exist_ok=True)
       filter = extractfilter(image,filt)
       padd = padding(image,filter)
       convolvedimage = convolution(padd,filter)
       save_path = os.path.join(
    filter_folder,
    name + "_" + filt + ".jpg"
)

       if filt in ["mean", "median", "gaussianblur"]:
        # These are intensity images → clip to [0, 255]
           display_image = np.clip(convolvedimage, 0, 255).astype(np.uint8)
       else:
        # Edge filters → preserve negative and positive responses
        # by normalizing for display
           display_image = to_displayable(convolvedimage)

       cv2.imwrite(save_path, display_image)
       cv2_imshow(display_image)
    # Print image information
    print("\nImage:", os.path.basename(path))
    print("Shape:", image.shape)
    print("Data type:", image.dtype)

    # Show image


In [ ]:
def gaussian_filter(sigma):
    size = 2*sigma + 1
    center = size // 2
    gaussian = np.zeros((size, size), dtype=float)

    for i in range(size):
        for j in range(size):
            x = i - center
            y = j - center

            gaussian[i, j] = np.exp(
                -(x*x + y*y) / (2 * sigma*sigma)
            )

    # Normalize so that all kernel values sum to 1
    gaussian = gaussian / gaussian.sum()

    return gaussian

In [ ]:
import copy
def gaussianunblur(image,max_iterations=20, tolerance=0.001):

  filter = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=float) / 16
  I0 = image.astype(float)
  I02 = copy.deepcopy(I0)

  for i in range(max_iterations):
    print(i+1)
    I1 = padding(I02,filter)
    I = copy.deepcopy(I1)
    A = convolution(I,filter)
    B = I0/(A + 1e-8)
    B1 = padding(B,filter)
    C = convolution(B1,filter)
    Inew = I02*C
    change = np.mean(np.abs(Inew - I02))
    relative_change = change / (np.mean(np.abs(I02)) + 1e-8)
    if relative_change < tolerance:
      print("Converged at iteration:", i + 1)
      I02 = Inew
      break
    I02 = Inew
  return I02

In [ ]:
import copy
def gaussianunblurwithsigma(image,sigma,max_iterations=50, tolerance=0.001):

  filter = gaussian_filter(sigma)
  I0 = image.astype(float)
  I02 = copy.deepcopy(I0)

  for i in range(max_iterations):
    print(i+1)
    I1 = padding(I02,filter)
    I = copy.deepcopy(I1)
    A = convolution(I,filter)
    B = I0/(A + 1e-8)
    B1 = padding(B,filter)
    C = convolution(B1,filter)
    Inew = I02*C
    change = np.mean(np.abs(Inew - I02))
    relative_change = change / (np.mean(np.abs(I02)) + 1e-8)
    if relative_change < tolerance:
      print("Converged at iteration:", i + 1)
      I02 = Inew
      break
    I02 = Inew
  return I02

In [ ]:
input_path = "/content"
image_paths = glob.glob(os.path.join(input_path, "*.jpg"))

print("Images found:", len(image_paths))
image_paths = [image_paths[0]]
k=0
# Read and extract each image
gaussblur_folder = os.path.join(output_path,  "gaussianunblur")
os.makedirs(gaussblur_folder, exist_ok=True)
for path in image_paths:
    k = k+1
    print(k)
    name = os.path.splitext(os.path.basename(path))[0]
    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        print("Could not read:", path)
        continue
    filter = extractfilter(image,"gaussianblur")
    padd = padding(image,filter)
    convolvedimage = convolution(padd,filter)
    unblurimage = gaussianunblur(convolvedimage)
    unblurimage = np.clip(
            unblurimage,
            0,
            255
        ).astype(np.uint8)
    save_path = os.path.join(
        gaussblur_folder,
        name + "_fixed_gaussian_unblur.jpg"
    )
    cv2.imwrite(save_path, unblurimage)
    cv2_imshow(unblurimage)
    # Print image information
    print("\nImage:", os.path.basename(path))
    print("Shape:", image.shape)
    print("Data type:", image.dtype)

    # Show image


In [ ]:
input_path = "/content"
image_paths = glob.glob(os.path.join(input_path, "*.jpg"))

print("Images found:", len(image_paths))
image_paths = [image_paths[0]]
k=0
# Read and extract each image
gaussblur_folder = os.path.join(output_path,  "gaussianunblur")
os.makedirs(gaussblur_folder, exist_ok=True)
for path in image_paths:
  for sigma in [1,2,3]:
    k = k+1
    print(k)
    name = os.path.splitext(os.path.basename(path))[0]
    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        print("Could not read:", path)
        continue
    filter = gaussian_filter(sigma)
    padd = padding(image,filter)
    convolvedimage = convolution(padd,filter)
    unblurimage = gaussianunblurwithsigma(convolvedimage,sigma)
    unblurimage = np.clip(
            unblurimage,
            0,
            255
        ).astype(np.uint8)

        # Save with sigma in filename
    save_path = os.path.join(
            gaussblur_folder,
            name + f"_sigma{sigma}_unblur.jpg"
        )

    cv2.imwrite(save_path, unblurimage)
    cv2_imshow(unblurimage)
    # Print image information
    print("\nImage:", os.path.basename(path))
    print("Shape:", image.shape)
    print("Data type:", image.dtype)

    # Show image